In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import torch_geometric
from torch_geometric.nn import GCNConv
import torch
import torch.nn.functional as F

In [ ]:
train = pd.read_csv("data/train.txt", sep=" ", header=None)
train.columns = ["u", "v", "label"]
G = nx.Graph()
edges = train[train["label"] == 1][["u", "v"]].values
G.add_edges_from(edges)

In [ ]:
test = pd.read_csv("data/test.txt", sep=" ", header=None)
test.columns = ["u", "v"]

In [ ]:
node_info = pd.read_csv("data/node_information.csv", header=None)
node_info = node_info.rename(columns={0: "node"})
node_features = {
    int(row["node"]): row.drop("node").values
    for _, row in node_info.iterrows()
}

In [ ]:
class GNN_model(torch.nn.Module):
    """
    Define a Graph Convolution Network 
    """
    def __init__(self, num_layers, input_size, hidden_size, output_size, dropout):
        super(GNN_model, self).__init__()
        # Define GNN components
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels=input_size,out_channels=hidden_size))
        for i in range(num_layers-2):
            self.convs.append(GCNConv(in_channels=hidden_size,out_channels=hidden_size))
        self.convs.append(GCNConv(in_channels=hidden_size,out_channels=output_size))
        self.dropout = dropout 

    def forward(self, graph, x=None):
        if x is None:
            x = graph.x
        edge_index = graph.edge_index
        for conv in self.convs[:-1]:
            x = conv(x,edge_index)
            x = F.relu(x)
            x = F.dropout(x)

        x = self.convs[-1](x,edge_index)
        output = F.log_softmax(x)

        return output

In [ ]:
for u, v in G.edges():
    x_u = node_features[u]
    x_v = node_features[v]

    sim = np.dot(x_u, x_v)
    G[u][v]['weight'] = 1 + 0.1 * sim